# 00 Setup and stage (CPU runtime)

Runtime: **CPU**. Mounts Drive, installs the repo, verifies the BIRD files and stages databases to the local disk.
Local disk is wiped when the runtime disconnects; staging is idempotent, so rerunning is cheap.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

## 1. Inspect the real BIRD files

Prints zip layout, JSON keys, gold file format, counts, JSON SQL vs gold, mini-dev vs dev, description diffs and `dev_tied_append.json`.
The report is also saved under `586/effsql/cache/inspect/`.

In [ ]:
!effsql inspect

## 2. Stage dev databases (mini-dev reuses them) and build the schema cache

In [ ]:
!effsql stage --split dev
!effsql stage --split minidev
!effsql schema-cache --split dev

## 3. Train databases, one shard at a time

The train zip is 9.3 GB compressed. Stage and cache shards of the 69 databases; the same `--shard i/n` selects the same databases in every command.
The schema cache lands on Drive, so later GPU sessions never need staged databases.

In [ ]:
SHARD = "0/8"
!effsql stage --split train --shard {SHARD}
!effsql schema-cache --split train --shard {SHARD}